In [2]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np
import yaml

from src.utils.paths import load_paths
from src.utils.logging import setup_logger

from src.datasets.pcap_reader import iter_packets
from src.flow.builder import FlowBuilder

In [3]:
paths = load_paths()
paths.ensure_dirs()
logger = setup_logger(level="INFO")

logger.info(f"Repo root: {paths.repo_root}")
logger.info(f"ISCX raw dir: {paths.data_raw / 'iscx'}")
logger.info(f"Processed dir: {paths.data_processed}")

2026-02-15 07:27:17 | INFO | ai-vpn-firewall | Repo root: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
2026-02-15 07:27:17 | INFO | ai-vpn-firewall | ISCX raw dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\raw\iscx
2026-02-15 07:27:17 | INFO | ai-vpn-firewall | Processed dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed


In [4]:
features_path = paths.configs_dir / "features.yaml"
assert features_path.exists(), f"Missing: {features_path}"

features_cfg = yaml.safe_load(features_path.read_text()) or {}
w = features_cfg.get("window") or {}

N = int(w.get("N", 100))
EPS = float(w.get("eps", 1e-6))
MIN_PACKETS = int(w.get("min_packets", 10))

logger.info(f"Loaded features config: N={N}, eps={EPS}, min_packets={MIN_PACKETS}")

2026-02-15 07:27:17 | INFO | ai-vpn-firewall | Loaded features config: N=100, eps=1e-06, min_packets=3


In [5]:
raw_iscx = paths.data_raw / "iscx"
vpn_dir = raw_iscx / "vpn"
nonvpn_dir = raw_iscx / "nonvpn"

assert vpn_dir.exists(), f"Missing folder: {vpn_dir}"
assert nonvpn_dir.exists(), f"Missing folder: {nonvpn_dir}"

def list_pcaps(d: Path):
    exts = {".pcap", ".pcapng"}
    return sorted([p for p in d.rglob("*") if p.suffix.lower() in exts])

vpn_pcaps = list_pcaps(vpn_dir)
nonvpn_pcaps = list_pcaps(nonvpn_dir)

logger.info(f"VPN pcaps: {len(vpn_pcaps)}")
logger.info(f"NonVPN pcaps: {len(nonvpn_pcaps)}")

vpn_pcaps[:5], nonvpn_pcaps[:5]

2026-02-15 07:27:17 | INFO | ai-vpn-firewall | VPN pcaps: 31
2026-02-15 07:27:17 | INFO | ai-vpn-firewall | NonVPN pcaps: 23


([WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1b.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_bittorrent.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_email2a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_email2b.pcap')],
 [WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/aim_chat_3a.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/aim_chat_3b.pcap'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/AIMchat1.pcapng'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonvpn/AIMchat2.pcapng'),
  WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/nonv

In [6]:
def make_non_decreasing(ts, eps: float):
    out = []
    prev = None
    for t in ts:
        t = float(t)
        if prev is None:
            out.append(t)
            prev = t
            continue
        if t < prev:
            t = prev + eps
        out.append(t)
        prev = t
    return out

def normalize_capture_name(name: str) -> str:
    s = str(name).strip().lower().replace("\\", "/").split("/")[-1]
    return s

def derive_app_from_prefixed_filename(fname: str) -> str:
    s = normalize_capture_name(fname)
    s = s.replace(".pcapng", "").replace(".pcap", "")
    s = s.replace("vpn_", "").replace("nonvpn_", "")
    return s.split("_", 1)[0]

In [7]:
test_pcap = nonvpn_pcaps[0]
logger.info(f"Testing one PCAP: {test_pcap}")

builder = FlowBuilder(inactivity_timeout=120.0)

n_pkts = 0
for rec in iter_packets(test_pcap):
    builder.add_packet(**rec)
    n_pkts += 1

flows_list = builder.finalize()
logger.info(f"Packets read: {n_pkts}")
logger.info(f"Flows built: {len(flows_list)}")

flows_list[0].keys(), flows_list[0]["connection"]

2026-02-15 07:27:17 | INFO | ai-vpn-firewall | Testing one PCAP: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\raw\iscx\nonvpn\aim_chat_3a.pcap
2026-02-15 07:27:20 | INFO | ai-vpn-firewall | Packets read: 1225
2026-02-15 07:27:20 | INFO | ai-vpn-firewall | Flows built: 40


(dict_keys(['connection', 'timestamps', 'sizes', 'directions']),
 ('131.202.240.87', 42534, '178.237.19.228', 443, 6))

In [8]:
rows = []

def process_pcap(pcap_path: Path, label: int):
    pref = "vpn_" if label == 1 else "nonvpn_"
    base = pcap_path.name
    file_name = pref + base

    builder = FlowBuilder(inactivity_timeout=120.0)

    pkt_count = 0
    for rec in iter_packets(pcap_path):
        builder.add_packet(**rec)
        pkt_count += 1

    built = builder.finalize()

    for f in built:
        ts = f["timestamps"]
        sz = f["sizes"]
        dr = f["directions"]

        ts = make_non_decreasing(ts, eps=EPS)

        rows.append({
            "connection": f["connection"],
            "timestamps": ts,
            "sizes": sz,
            "directions": dr,
            "file_names": file_name,
            "label": int(label),
        })

    return pkt_count, len(built)

# Run
total_pkts = 0
total_flows = 0

for i, p in enumerate(nonvpn_pcaps):
    pk, fl = process_pcap(p, label=0)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"NonVPN processed: {i+1}/{len(nonvpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

for i, p in enumerate(vpn_pcaps):
    pk, fl = process_pcap(p, label=1)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"VPN processed: {i+1}/{len(vpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

logger.info(f"TOTAL pkts={total_pkts} TOTAL flows={total_flows}")
df = pd.DataFrame(rows)
df.head(), df.shape

2026-02-15 07:28:22 | INFO | ai-vpn-firewall | NonVPN processed: 10/23 | pkts=142984 flows=7929
2026-02-15 07:39:04 | INFO | ai-vpn-firewall | NonVPN processed: 20/23 | pkts=2648354 flows=88007
2026-02-15 07:40:59 | INFO | ai-vpn-firewall | VPN processed: 10/31 | pkts=3320126 flows=91565
2026-02-15 07:48:07 | INFO | ai-vpn-firewall | VPN processed: 20/31 | pkts=5655126 flows=103135
2026-02-15 07:53:31 | INFO | ai-vpn-firewall | VPN processed: 30/31 | pkts=7275809 flows=106394
2026-02-15 07:54:07 | INFO | ai-vpn-firewall | TOTAL pkts=7484145 TOTAL flows=106607


(                                        connection  \
 0  (131.202.240.87, 42534, 178.237.19.228, 443, 6)   
 1  (131.202.240.87, 42530, 178.237.17.103, 443, 6)   
 2   (205.188.12.91, 443, 131.202.240.87, 42522, 6)   
 3   (131.202.240.87, 51542, 224.0.0.252, 5355, 17)   
 4   (131.202.240.87, 60662, 224.0.0.252, 5355, 17)   
 
                                           timestamps  \
 0  [1430325788.00104, 1430325788.17371, 143032581...   
 1  [1430325788.00169, 1430325788.172106, 14303258...   
 2  [1430325803.394782, 1430325803.415641, 1430325...   
 3             [1430325829.348184, 1430325829.761916]   
 4             [1430325829.348542, 1430325829.761938]   
 
                                                sizes  \
 0  [60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 6...   
 1  [60, 60, 60, 60, 60, 60, 60, 60, 269, 54, 60, ...   
 2  [331, 54, 155, 60, 203, 315, 155, 155, 54, 373...   
 3                                           [64, 64]   
 4                                     

In [10]:
df = df.reset_index(drop=True)
df["row_id"] = df.index.astype("int64")

expected_cols = {"connection", "timestamps", "sizes", "directions", "file_names", "label", "row_id"}
missing = expected_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"

def is_listlike(x): return isinstance(x, (list, tuple))

for col in ["timestamps", "sizes", "directions"]:
    bad = df[~df[col].map(is_listlike)]
    assert len(bad) == 0, f"Non-list entries in {col}: {bad.head()}"

lens = pd.DataFrame({
    "t": df["timestamps"].map(len),
    "s": df["sizes"].map(len),
    "d": df["directions"].map(len),
})
mismatch = df[(lens["t"] != lens["s"]) | (lens["t"] != lens["d"])]
assert len(mismatch) == 0, f"Mismatched list lengths:\n{mismatch.head()}"

bad_dir = df[~df["directions"].map(lambda dirs: set(dirs).issubset({0,1}))]
assert len(bad_dir) == 0, f"Invalid direction values:\n{bad_dir.head()}"

def sizes_valid(sz) -> bool:
    if any(x is None for x in sz):
        return False
    return all((isinstance(x, (int, float)) and x >= 0 and x < 65536) for x in sz)

bad_sizes = df[~df["sizes"].map(sizes_valid)]
assert len(bad_sizes) == 0, f"Invalid sizes:\n{bad_sizes[['file_names','sizes']].head()}"

df["file_names"] = df["file_names"].astype(str)
df["capture_name"] = df["file_names"].map(normalize_capture_name)
df["capture_id"] = df["capture_name"]

def conn_to_str(conn) -> str:
    try:
        src_ip, src_port, dst_ip, dst_port, proto = conn
        return f"{src_ip}:{int(src_port)}-{dst_ip}:{int(dst_port)}-p{int(proto)}"
    except Exception:
        return str(conn)

df["connection_str"] = df["connection"].map(conn_to_str)
df["flow_key"] = df["connection_str"]
df["flow_id"] = df["capture_id"] + "::" + df["row_id"].astype(str)

df["app"] = df["file_names"].map(derive_app_from_prefixed_filename)

df["packet_count_full"] = df["sizes"].map(len)

df["timestamps"] = df["timestamps"].map(lambda xs: xs[:N])
df["sizes"] = df["sizes"].map(lambda xs: xs[:N])
df["directions"] = df["directions"].map(lambda xs: xs[:N])

df["packet_count"] = df["sizes"].map(len)
df["window_complete"] = df["packet_count_full"] >= N
df["min_packets_ok"] = df["packet_count"] >= MIN_PACKETS

logger.info(f"ISCX flows built: shape={df.shape}")
logger.info("Label counts:\n" + str(df["label"].value_counts()))
logger.info(f"min_packets_ok rate: {100*df['min_packets_ok'].mean():.2f}%")

2026-02-15 22:37:36 | INFO | ai-vpn-firewall | ISCX flows built: shape=(106607, 17)
2026-02-15 22:37:36 | INFO | ai-vpn-firewall | Label counts:
label
0    88139
1    18468
Name: count, dtype: int64
2026-02-15 22:37:36 | INFO | ai-vpn-firewall | min_packets_ok rate: 4.38%


In [11]:
flows = df[
    [
        "capture_id",
        "capture_name",
        "row_id",
        "flow_id",
        "flow_key",
        "connection_str",
        "timestamps",
        "sizes",
        "directions",
        "file_names",
        "app",
        "label",
        "packet_count",
        "packet_count_full",
        "window_complete",
        "min_packets_ok",
    ]
].copy()

assert flows["flow_id"].is_unique

flows["timestamps"] = flows["timestamps"].map(lambda xs: [float(x) for x in xs])
flows["sizes"] = flows["sizes"].map(lambda xs: [int(x) for x in xs])
flows["directions"] = flows["directions"].map(lambda xs: [int(x) for x in xs])

out_dir = paths.data_processed / "iscx"
out_dir.mkdir(parents=True, exist_ok=True)

flows_path = out_dir / "flows.parquet"
flows.to_parquet(flows_path, index=False)

logger.info(f"Saved ISCX flows parquet: {flows_path}")

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "dataset": "iscx",
    "flows_parquet": str(flows_path),
    "flows_sha256": sha256_file(flows_path),
    "features_yaml": str(features_path),
    "features_yaml_sha256": hashlib.sha256(features_path.read_bytes()).hexdigest(),
    "rows": int(len(flows)),
    "unique_captures": int(flows["capture_id"].nunique()),
    "unique_flows": int(flows["flow_id"].nunique()),
    "label_counts": flows["label"].value_counts().to_dict(),
    "window": {"N": int(N), "eps": float(EPS), "min_packets": int(MIN_PACKETS)},
    "pct_window_complete": float((flows["packet_count_full"] >= N).mean() * 100),
    "pct_min_packets_ok": float((flows["min_packets_ok"]).mean() * 100),
}

manifest_path = out_dir / "flows_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
logger.info(f"Saved ISCX flows manifest: {manifest_path}")

flows.head()

2026-02-15 22:38:26 | INFO | ai-vpn-firewall | Saved ISCX flows parquet: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\flows.parquet
2026-02-15 22:38:26 | INFO | ai-vpn-firewall | Saved ISCX flows manifest: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\flows_manifest.json


,capture_id,capture_name,row_id,flow_id,flow_key,connection_str,timestamps,sizes,directions,file_names,app,label,packet_count,packet_count_full,window_complete,min_packets_ok
0,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,0,nonvpn_aim_chat_3a.pcap::0,131.202.240.87:42534-178.237.19.228:443-p6,131.202.240.87:42534-178.237.19.228:443-p6,"[1430325788.00104, 1430325788.17371, 143032581...","[60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 6...","[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, ...",nonvpn_aim_chat_3a.pcap,nonaim,0,42,42,False,True
1,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,1,nonvpn_aim_chat_3a.pcap::1,131.202.240.87:42530-178.237.17.103:443-p6,131.202.240.87:42530-178.237.17.103:443-p6,"[1430325788.00169, 1430325788.172106, 14303258...","[60, 60, 60, 60, 60, 60, 60, 60, 269, 54, 60, ...","[1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, ...",nonvpn_aim_chat_3a.pcap,nonaim,0,44,44,False,True
2,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,2,nonvpn_aim_chat_3a.pcap::2,205.188.12.91:443-131.202.240.87:42522-p6,205.188.12.91:443-131.202.240.87:42522-p6,"[1430325803.394782, 1430325803.415641, 1430325...","[331, 54, 155, 60, 203, 315, 155, 155, 54, 373...","[1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, ...",nonvpn_aim_chat_3a.pcap,nonaim,0,100,299,True,True
3,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,3,nonvpn_aim_chat_3a.pcap::3,131.202.240.87:51542-224.0.0.252:5355-p17,131.202.240.87:51542-224.0.0.252:5355-p17,"[1430325829.348184, 1430325829.761916]","[64, 64]","[1, 1]",nonvpn_aim_chat_3a.pcap,nonaim,0,2,2,False,False
4,nonvpn_aim_chat_3a.pcap,nonvpn_aim_chat_3a.pcap,4,nonvpn_aim_chat_3a.pcap::4,131.202.240.87:60662-224.0.0.252:5355-p17,131.202.240.87:60662-224.0.0.252:5355-p17,"[1430325829.348542, 1430325829.761938]","[64, 64]","[1, 1]",nonvpn_aim_chat_3a.pcap,nonaim,0,2,2,False,False


In [12]:
from src.features.extract import load_feature_config, extract_features_from_flows, feature_config_hash_text
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts

features_yaml = paths.configs_dir / "features.yaml"
cfg = load_feature_config(features_yaml)

flows = pd.read_parquet(flows_path)
logger.info(f"Loaded ISCX flows: {flows.shape}")

logger.info("Extracting ISCX features...")
features_raw = extract_features_from_flows(flows=flows, cfg=cfg)
logger.info(f"Raw ISCX features: {features_raw.shape}")

art = default_feature_artifacts(paths.artifacts_features)


pipe = None
if hasattr(FeaturePipeline, "load"):
    pipe = FeaturePipeline.load(art)
elif hasattr(FeaturePipeline, "load_from_artifacts"):
    pipe = FeaturePipeline.load_from_artifacts(art)
else:
    pipe = FeaturePipeline()
    if hasattr(pipe, "load"):
        pipe = pipe.load(art)
    else:
        raise RuntimeError("FeaturePipeline has no load method. Show me feature_pipeline.py and I’ll adapt this cell.")

features_scaled = pipe.transform(features_raw)

features_scaled["split"] = "iscx_test"

features_out = out_dir / "features.parquet"
features_scaled.to_parquet(features_out, index=False)
logger.info(f"Saved ISCX scaled features: {features_out}")

features_scaled.head()

2026-02-15 22:38:40 | INFO | ai-vpn-firewall | Loaded ISCX flows: (106607, 16)
2026-02-15 22:38:40 | INFO | ai-vpn-firewall | Extracting ISCX features...
2026-02-15 22:40:11 | INFO | ai-vpn-firewall | Raw ISCX features: (106607, 91)
2026-02-15 22:40:12 | INFO | ai-vpn-firewall | Saved ISCX scaled features: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\features.parquet


,flow_id,capture_id,label,f_duration_s,f_total_pkts,f_up_pkts,f_down_pkts,f_total_bytes,f_up_bytes,f_down_bytes,...,h_iat_all_06,h_iat_all_07,h_iat_all_08,h_iat_all_09,h_iat_all_10,h_iat_all_11,q_packet_count,q_window_complete,q_min_packets_ok,split
0,nonvpn_aim_chat_3a.pcap::0,nonvpn_aim_chat_3a.pcap,0,-0.190200,0.508469,0.500918,0.483905,-0.349710,-0.217956,-0.296443,...,-0.600885,-0.215298,42.166751,-0.213712,-0.196717,1.582959,42.0,0.0,1.0,iscx_test
1,nonvpn_aim_chat_3a.pcap::1,nonvpn_aim_chat_3a.pcap,0,-0.190201,0.568548,0.561034,0.540211,-0.333138,-0.212367,-0.279515,...,-0.600885,1.005961,40.190138,-0.213712,-0.196717,1.585518,44.0,0.0,1.0,iscx_test
2,nonvpn_aim_chat_3a.pcap::2,nonvpn_aim_chat_3a.pcap,0,-0.286581,2.250758,2.304401,2.060474,0.205476,0.503342,-0.053968,...,3.915866,9.863177,-0.330439,3.303227,1.282893,0.567165,100.0,1.0,1.0,iscx_test
3,nonvpn_aim_chat_3a.pcap::3,nonvpn_aim_chat_3a.pcap,0,-0.317741,-0.693109,-0.641287,-0.698522,-0.472434,-0.335136,-0.375737,...,-0.600885,-0.215298,-0.330439,69.421677,-0.196717,-0.617557,2.0,0.0,0.0,iscx_test
4,nonvpn_aim_chat_3a.pcap::4,nonvpn_aim_chat_3a.pcap,0,-0.317741,-0.693109,-0.641287,-0.698522,-0.472434,-0.335136,-0.375737,...,-0.600885,-0.215298,-0.330439,69.421677,-0.196717,-0.617557,2.0,0.0,0.0,iscx_test


In [13]:
def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

features_manifest = {
    "dataset": "iscx",
    "flows_parquet": str(flows_path.resolve()),
    "flows_sha256": sha256_file(flows_path),
    "features_parquet": str(features_out.resolve()),
    "features_sha256": sha256_file(features_out),
    "features_yaml": str(features_yaml.resolve()),
    "features_yaml_sha256": hashlib.sha256(features_yaml.read_bytes()).hexdigest(),
    "rows": {
        "flows": int(len(flows)),
        "features": int(len(features_scaled)),
        "trainable_min_packets_ok": int((features_scaled["q_min_packets_ok"] == 1.0).sum()),
    },
    "label_counts": features_scaled["label"].value_counts().to_dict(),
    "schema": {
        "n_columns": int(features_scaled.shape[1]),
        "columns": list(features_scaled.columns),
        "dtypes": {c: str(features_scaled[c].dtype) for c in features_scaled.columns},
    },
}

features_manifest_path = out_dir / "features_manifest.json"
features_manifest_path.write_text(json.dumps(features_manifest, indent=2), encoding="utf-8")
logger.info(f"Saved ISCX features manifest: {features_manifest_path}")

print(json.dumps({
    "features_rows": features_manifest["rows"]["features"],
    "trainable_rows": features_manifest["rows"]["trainable_min_packets_ok"],
    "features_sha256": features_manifest["features_sha256"],
}, indent=2))

2026-02-15 22:40:13 | INFO | ai-vpn-firewall | Saved ISCX features manifest: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed\iscx\features_manifest.json
{
  "features_rows": 106607,
  "trainable_rows": 4671,
  "features_sha256": "352582988d01f5a89890d0073303b4f302b4db469cd2a2f3227599834dd42c54"
}
